# Matching ratio, feature families, cross-evaluation and horizons (full window)

Reports landmark-time and stay-length percentiles, the matching-ratio sensitivity, feature-family models, SIRS and qSOFA, cross-evaluation between designs and the 4-h and 12-h horizons.

Data: PhysioNet/Computing in Cardiology Challenge 2019 training sets A and B (open access, CC BY 4.0), downloaded by the first code cell.

Run on Google Colab with a CPU runtime (Runtime > Run all). All results are written to `results_03.json`, which is downloaded at the end.

## Download the data

In [ ]:
import os, glob
!pip -q install awscli
!aws s3 sync --no-sign-request --only-show-errors s3://physionet-open/challenge-2019/1.0.0/training/ /content/data/
nA = len(glob.glob('/content/data/training_setA/*.psv')); nB = len(glob.glob('/content/data/training_setB/*.psv'))
print('Set A files:', nA, ' Set B files:', nB, ' (expected 20336 and 20000)')
if nA < 20336 or nB < 20000:
    print('S3 download incomplete - using wget instead (slower)')
    !wget -r -N -c -np -q https://physionet.org/files/challenge-2019/1.0.0/training/ -P /content/

## Load all ICU stays

In [ ]:
import os, glob, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
DATA = None
for cand in ['/content/data', '/content/physionet.org/files/challenge-2019/1.0.0/training']:
    if cand and glob.glob(f'{cand}/training_setA/*.psv'):
        DATA = cand; break
assert DATA, 'PhysioNet files not found - run the download cell first'
print('Using data in', DATA)
DYN = ['HR','O2Sat','Temp','SBP','MAP','DBP','Resp','EtCO2','BaseExcess','HCO3','FiO2','pH','PaCO2','SaO2',
       'AST','BUN','Alkalinephos','Calcium','Chloride','Creatinine','Bilirubin_direct','Glucose','Lactate',
       'Magnesium','Phosphate','Potassium','Bilirubin_total','TroponinI','Hct','Hgb','PTT','WBC','Fibrinogen','Platelets']
H = 6          # prediction horizon (hours)
W = 6          # feature window (hours)
W_MIN = 6      # minimum observed hours before a landmark
K_CTRL = 10    # controls per septic stay in risk-set sampling
SEEDS = [2026, 2027, 2028, 2029, 2030]
OUT = {}

def load_set(s):
    files = sorted(glob.glob(f'{DATA}/training_set{s}/*.psv'))
    stays = {}
    t0 = time.time()
    for f in files:
        d = pd.read_csv(f, sep='|')
        pid = os.path.basename(f)[:-4]
        lab = d['SepsisLabel'].to_numpy()
        pos = np.flatnonzero(lab == 1)
        stays[pid] = dict(X=d[DYN].to_numpy(np.float32), n=len(d),
                          first=(int(pos[0]) if len(pos) else -1),
                          age=float(d['Age'].iloc[0]), sex=float(d['Gender'].iloc[0]),
                          hadm=float(d['HospAdmTime'].iloc[0]), icu0=float(d['ICULOS'].iloc[0]),
                          unit=(0 if d['Unit1'].iloc[0] == 1 else (1 if d['Unit2'].iloc[0] == 1 else 2)))
    print(f'Set {s}: {len(stays)} stays loaded in {time.time()-t0:.0f}s')
    return stays

S = {s: load_set(s) for s in ['A', 'B']}
assert len(S['A']) > 0 and len(S['B']) > 0, 'Data not found'

## Helpers

In [ ]:
def features(st, lm):
    F = np.full((len(lm), 7 * len(DYN) + 4), np.nan, np.float32)
    meta = np.zeros((len(lm), 4), np.float32)   # age, sex, unit, tp (not model features)
    for r, (pid, y, tp) in enumerate(lm.itertuples(index=False)):
        v = st[pid]; hist = v['X'][:tp + 1]; win = hist[max(0, tp - W + 1):]
        with np.errstate(all='ignore'):
            ok = ~np.isnan(win); cnt = ok.sum(0)
            mean = np.nanmean(win, 0); sd = np.nanstd(win, 0); mn = np.nanmin(win, 0); mx = np.nanmax(win, 0)
            okh = ~np.isnan(hist)
            idx = np.where(okh.any(0), hist.shape[0] - 1 - np.argmax(okh[::-1], 0), -1)
            last = np.where(idx >= 0, hist[np.maximum(idx, 0), np.arange(hist.shape[1])], np.nan)
            t = np.arange(win.shape[0], dtype=np.float32)[:, None]
            tm = np.where(ok, t, 0).sum(0) / np.maximum(cnt, 1); vm = np.where(ok, win, 0).sum(0) / np.maximum(cnt, 1)
            cov = np.where(ok, (t - tm) * (np.nan_to_num(win) - vm), 0).sum(0)
            var = np.where(ok, (t - tm) ** 2, 0).sum(0)
            slope = np.where((cnt >= 2) & (var > 0), cov / np.where(var > 0, var, 1), np.nan)
            miss = 1 - cnt / win.shape[0]
        F[r, :7 * len(DYN)] = np.concatenate([mean, sd, mn, mx, last, slope, miss])
        F[r, -4:] = [v['age'], v['sex'], v['hadm'], miss.mean()]
        meta[r] = [v['age'], v['sex'], v['unit'], tp]
    return F, meta

def hgb(seed):
    return HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                          max_depth=6, l2_regularization=1.0, class_weight='balanced',
                                          random_state=seed)

def wq_thresholds(cal_s, cal_w, test_w, alpha):
    """Weighted conformal quantile (Tibshirani et al. 2019) for each test point; inf if not attainable."""
    o = np.argsort(cal_s); s = cal_s[o]; cw = np.cumsum(cal_w[o]); tot = cw[-1]
    need = (1 - alpha) * (tot + test_w)
    k = np.searchsorted(cw, need, side='left')
    thr = np.full(len(test_w), np.inf); ok = k < len(s); thr[ok] = s[k[ok]]
    return thr

## Cohort builders

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def septic_cases(st, h):
    rows = []
    for pid, v in st.items():
        if v['first'] < 0: continue
        tp = v['first'] + 6 - h
        if W_MIN - 1 <= tp <= v['n'] - 1: rows.append((pid, 1, tp))
    return rows

def lm_trunc(st, h, seed):
    rng = np.random.default_rng(seed); cases = septic_cases(st, h); ref = np.array([c[2] for c in cases])
    rows = list(cases)
    for pid, v in st.items():
        if v['first'] >= 0: continue
        cand = ref[ref <= v['n'] - 1]
        if len(cand): rows.append((pid, 0, int(rng.choice(cand))))
    return pd.DataFrame(rows, columns=['pid', 'y', 'tp']), 0

def lm_riskset(st, h, seed, k):
    rng = np.random.default_rng(seed); cases = septic_cases(st, h)
    neg_pid = np.array([p for p, v in st.items() if v['first'] < 0]); neg_n = np.array([st[p]['n'] for p in neg_pid])
    rows, dropped = [], 0
    for pid, _, t in cases:
        pool = np.flatnonzero(neg_n - 1 >= t + h)
        if len(pool) < k: dropped += 1; continue
        rows.append((pid, 1, t)); rows += [(neg_pid[j], 0, t) for j in rng.choice(pool, size=k, replace=False)]
    return pd.DataFrame(rows, columns=['pid', 'y', 'tp']), dropped

def build(name, h, k=None):
    out = {}
    for s in ['A', 'B']:
        seed = 2019 + h + (0 if s == 'A' else 100)
        lm, dropped = lm_trunc(S[s], h, seed) if name == 'trunc' else lm_riskset(S[s], h, seed, k)
        X, meta = features(S[s], lm)
        out[s] = dict(X=X, y=lm.y.to_numpy(), tp=lm.tp.to_numpy(), pid=lm.pid.to_numpy(), dropped=int(dropped))
    return out

## Timing diagnostics

In [ ]:
tdiag = {}
for s in ['A', 'B']:
    sep_tp = np.array([c[2] for c in septic_cases(S[s], H)])
    neg_n = np.array([v['n'] for v in S[s].values() if v['first'] < 0])
    q = [10, 25, 50, 75, 90, 99]
    tdiag[s] = dict(septic_landmark_quantiles=dict(zip(map(str, q), np.percentile(sep_tp, q).tolist())),
                    nonseptic_length_quantiles=dict(zip(map(str, q), np.percentile(neg_n, q).tolist())),
                    nonseptic_max_length=int(neg_n.max()),
                    frac_septic_landmark_beyond_nonseptic_p99=float(np.mean(sep_tp + H > np.percentile(neg_n, 99))))
OUT['timing_diag'] = tdiag
print(json.dumps(tdiag, indent=1))

## Robustness: matching ratio, within- vs cross-hospital, feature families, SIRS/qSOFA

In [ ]:
NC = len(DYN)
VIT = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'EtCO2']
LAB = [c for c in DYN if c not in VIT]
blk = lambda b, cols: [b * NC + DYN.index(c) for c in cols]
FAM = {'all': None,
       'vitals_values': sum((blk(b, VIT) for b in range(6)), []),
       'labs_values': sum((blk(b, LAB) for b in range(6)), []),
       'missingness_only': blk(6, DYN) + [7 * NC + 3],
       'demographics_only': [7 * NC, 7 * NC + 1, 7 * NC + 2]}

def last(X, c): return X[:, 4 * NC + DYN.index(c)]
def sirs(X):
    z = lambda c: np.nan_to_num(last(X, c), nan=-1e9)
    zt = last(X, 'Temp')
    return ((z('HR') > 90).astype(int) + (((zt > 38) | (zt < 36)) & ~np.isnan(zt)).astype(int)
            + ((z('Resp') > 20) | ((last(X, 'PaCO2') < 32) & ~np.isnan(last(X, 'PaCO2')))).astype(int)
            + (((last(X, 'WBC') > 12) | (last(X, 'WBC') < 4)) & ~np.isnan(last(X, 'WBC'))).astype(int)).astype(float)
def qsofa(X):
    rr = last(X, 'Resp'); sbp = last(X, 'SBP')
    return (((rr >= 22) & ~np.isnan(rr)).astype(int) + ((sbp <= 100) & ~np.isnan(sbp)).astype(int)).astype(float)

def stay_split(y, pid, frac, seed):
    rng = np.random.default_rng(seed); te = np.zeros(len(y), bool)
    for lab in (0, 1):
        u = np.unique(pid[y == lab]); pick = rng.choice(u, int(round(frac * len(u))), replace=False)
        te |= np.isin(pid, pick)
    # a control stay used in several risk sets must not appear on both sides
    te = np.isin(pid, np.unique(pid[te]))
    return ~te, te

def boot_auc(y, p, pid, B=200, seed=0):
    rng = np.random.default_rng(seed); code = pd.factorize(pid)[0]; order = np.argsort(code, kind='stable')
    bounds = np.r_[0, np.cumsum(np.bincount(code))]; n_u = len(bounds) - 1; a = []
    for _ in range(B):
        u = rng.integers(0, n_u, n_u); i = np.concatenate([order[bounds[j]:bounds[j + 1]] for j in u])
        if y[i].min() != y[i].max(): a.append(roc_auc_score(y[i], p[i]))
    return [float(np.percentile(a, 2.5)), float(np.percentile(a, 97.5))]

def settings(C):
    """A->B, B->A transfer and A->A, B->B within-hospital (30% of stays held out)."""
    out = {}
    for src, tgt in (('A', 'B'), ('B', 'A')):
        out[f'{src}to{tgt}'] = (C[src]['X'], C[src]['y'], C[tgt]['X'], C[tgt]['y'], C[tgt]['pid'])
    for s in ('A', 'B'):
        tr, te = stay_split(C[s]['y'], C[s]['pid'], 0.3, 2026)
        out[f'{s}to{s}'] = (C[s]['X'][tr], C[s]['y'][tr], C[s]['X'][te], C[s]['y'][te], C[s]['pid'][te])
    return out

def fit_eval(Xtr, ytr, Xte, yte, cols=None, ci=None):
    keep = (~np.isnan(Xtr)).any(0) & (~np.isnan(Xte)).any(0)
    if cols is not None:
        m_ = np.zeros(Xtr.shape[1], bool); m_[cols] = True; keep &= m_
    m = hgb(2026).fit(Xtr[:, keep], ytr); p = m.predict_proba(Xte[:, keep])[:, 1]
    r = dict(AUROC=float(roc_auc_score(yte, p)), AUPRC=float(average_precision_score(yte, p)), prev=float(yte.mean()))
    if ci is not None: r['AUROC_CI'] = boot_auc(yte, p, ci)
    return r, m, keep

ROB = {}
t0 = time.time()
COH = {'trunc': build('trunc', H)}
for k in (1, 3, 5, 10):
    COH[f'riskset_k{k}'] = build('riskset', H, k)
for name, C in COH.items():
    ROB[name] = dict(dropped_A=C['A']['dropped'], dropped_B=C['B']['dropped'],
                     rows_A=int(len(C['A']['y'])), rows_B=int(len(C['B']['y'])))
    for sname, (Xtr, ytr, Xte, yte, pte) in settings(C).items():
        main_ci = pte if name in ('trunc', 'riskset_k10') else None
        r, _, _ = fit_eval(Xtr, ytr, Xte, yte, ci=main_ci)
        r['SIRS_AUROC'] = float(roc_auc_score(yte, sirs(Xte))); r['qSOFA_AUROC'] = float(roc_auc_score(yte, qsofa(Xte)))
        if name in ('trunc', 'riskset_k10'):
            for fam, cols in FAM.items():
                if fam == 'all': continue
                r[f'family_{fam}_AUROC'] = fit_eval(Xtr, ytr, Xte, yte, cols=cols)[0]['AUROC']
        ROB[name][sname] = r
        print(f"{name:12s} {sname}: AUROC {r['AUROC']:.3f}  SIRS {r['SIRS_AUROC']:.3f}  qSOFA {r['qSOFA_AUROC']:.3f}"
              + (f"  CI {r['AUROC_CI'][0]:.3f}-{r['AUROC_CI'][1]:.3f}" if 'AUROC_CI' in r else '')
              + f"  [{time.time()-t0:.0f}s]")
OUT['robustness'] = ROB

## Cross-evaluation between designs

In [ ]:
CROSS = {}
for src, tgt in (('A', 'B'), ('B', 'A')):
    for tr_name, te_name in (('trunc', 'riskset_k10'), ('riskset_k10', 'trunc')):
        Ctr, Cte = COH[tr_name][src], COH[te_name][tgt]
        r = fit_eval(Ctr['X'], Ctr['y'], Cte['X'], Cte['y'])[0]
        CROSS[f'train_{tr_name}_{src}_test_{te_name}_{tgt}'] = r
        print(f"train {tr_name:11s} on {src} -> test {te_name:11s} on {tgt}: AUROC {r['AUROC']:.3f}")
OUT['cross_eval'] = CROSS

## Horizons 4 h and 12 h

In [ ]:
HOR = {}
for h in (4, 12):
    C = build('riskset', h, 10)
    HOR[str(h)] = dict(dropped_A=C['A']['dropped'], dropped_B=C['B']['dropped'])
    for sname, (Xtr, ytr, Xte, yte, pte) in settings(C).items():
        r = fit_eval(Xtr, ytr, Xte, yte)[0]; r['SIRS_AUROC'] = float(roc_auc_score(yte, sirs(Xte)))
        HOR[str(h)][sname] = r
        print(f"h={h:2d} {sname}: AUROC {r['AUROC']:.3f}  SIRS {r['SIRS_AUROC']:.3f}")
OUT['horizons_riskset_k10'] = HOR

## Save results

In [ ]:
with open('results_03.json', 'w') as fh:
    json.dump(OUT, fh, indent=1)
print('Saved results_03.json')

In [ ]:
from google.colab import files
files.download('results_03.json')